Fluxo de trabalho:
Imagem > Detecção > Classificação de componentes > Mapeamento STRIDE > Relatório em Markdown

In [7]:
import sys
from pathlib import Path

# adiciona a raiz do repo no PYTHONPATH
repo_root = Path.cwd().parent  # notebooks/ -> repo root
sys.path.append(str(repo_root))

print("repo_root:", repo_root)
print("src exists:", (repo_root / "src").exists())

from src.report import render_markdown, export_markdown

# Paths
MODEL_PATH = Path("runs/detect/train/weights/best.pt")   # depois do treino
EVAL_DIR = Path("data/eval/images")
OUT_DIR = Path("assets/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CATALOG_PATH = Path("data/catalog/stride_catalog.yaml")

print("Eval dir exists:", EVAL_DIR.exists())
print("Catalog exists:", CATALOG_PATH.exists())
print("Eval images:", [p.name for p in sorted(EVAL_DIR.glob('*.png'))])

repo_root: /Users/vilella/Documents/fiap/pos_ia/iadt_fase05_final
src exists: True
Eval dir exists: False
Catalog exists: False
Eval images: []


In [6]:
from src.report import render_markdown, export_markdown

In [1]:
import sys
from pathlib import Path
from ultralytics import YOLO

REPO_ROOT = Path.cwd().parent
EVAL_DIR = REPO_ROOT / "data" / "eval" / "images"
OUT_DIR = REPO_ROOT / "assets" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = REPO_ROOT / "runs" / "detect_train_mvp3" / "weights" / "best.pt"
print("Model exists:", MODEL_PATH.exists())

images = sorted(list(EVAL_DIR.glob("*.png")) + list(EVAL_DIR.glob("*.jpg")))
print("Eval images:", [p.name for p in images])

model = YOLO(str(MODEL_PATH))

# salva imagens com bounding boxes em runs/predict..., e também copia para assets/outputs
results = model.predict([str(p) for p in images], conf=0.25, save=True)

print("Predict done.")

Model exists: True
Eval images: ['arq1.png', 'arq2.png']

0: 640x640 (no detections), 39.2ms
1: 640x640 1 database, 39.2ms
Speed: 2.8ms preprocess, 39.2ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /Users/vilella/Documents/fiap/pos_ia/iadt_fase05_final/notebooks/runs/detect/predict
Predict done.


In [2]:
from pathlib import Path
from ultralytics import YOLO

REPO_ROOT = Path.cwd().parent
EVAL_DIR = REPO_ROOT / "data" / "eval" / "images"
MODEL_PATH = REPO_ROOT / "runs" / "detect_train_mvp3" / "weights" / "best.pt"

model = YOLO(str(MODEL_PATH))
images = sorted(list(EVAL_DIR.glob("*.png")) + list(EVAL_DIR.glob("*.jpg")))

results = model.predict([str(p) for p in images], conf=0.10, imgsz=1280, save=True)
print("Done.")


0: 1280x1280 3 gateways, 1 database, 163.4ms
1: 1280x1280 3 services, 6 databases, 163.4ms
Speed: 8.3ms preprocess, 163.4ms inference, 0.5ms postprocess per image at shape (1, 3, 1280, 1280)
Results saved to /Users/vilella/Documents/fiap/pos_ia/iadt_fase05_final/notebooks/runs/detect/predict2
Done.


In [6]:
from collections import Counter

counts = Counter()
top = {}

for r in results:
    names = r.names
    if r.boxes is None:
        continue
    for b in r.boxes:
        cls_id = int(b.cls[0].item())
        label = names[cls_id]
        conf = float(b.conf[0].item())
        counts[label] += 1
        top[label] = max(top.get(label, 0.0), conf)

print("Counts per class:", counts)
print("Top confidence per class:", top)

Counts per class: Counter({'database': 7, 'gateway': 3, 'service': 3})
Top confidence per class: {'database': 0.9894142150878906, 'gateway': 0.3168959617614746, 'service': 0.8785296678543091}


In [7]:
import sys
from pathlib import Path
import yaml

REPO_ROOT = Path.cwd().parent
sys.path.append(str(REPO_ROOT))

from src.report import render_markdown, export_markdown

CATALOG_PATH = REPO_ROOT / "data" / "catalog" / "stride_catalog.yaml"
OUT_DIR = REPO_ROOT / "assets" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(CATALOG_PATH, "r") as f:
    catalog = yaml.safe_load(f)

# transforma results (ultralytics) -> detections simples
detections = []
for r in results:
    names = r.names  # dict id->name
    if r.boxes is None:
        continue
    for b in r.boxes:
        cls_id = int(b.cls[0].item())
        conf = float(b.conf[0].item())
        xyxy = [float(x) for x in b.xyxy[0].tolist()]
        detections.append({
            "label": names[cls_id],
            "confidence": conf,
            "bbox": xyxy
        })

# gera threats por componente (MVP: agrega por tipo detectado)
threats = []
seen = sorted(set(d["label"] for d in detections))
for comp in seen:
    if comp not in catalog:
        continue
    for stride_cat, content in catalog[comp].items():
        threats.append({
            "component_label": comp,
            "stride_category": stride_cat,
            "vulnerabilities": content.get("vulns", []),
            "mitigations": content.get("mitigations", [])
        })

# adapters simples para o report.py (que usa getattr)
class Obj: 
    def __init__(self, **kw): self.__dict__.update(kw)

detections_obj = [Obj(**d) for d in detections]
threats_obj = [Obj(**t) for t in threats]

md = render_markdown(detections_obj, threats_obj)
out_path = OUT_DIR / "report.md"
export_markdown(md, str(out_path))

print("Detections:", len(detections_obj), "Components:", seen)
print("Saved:", out_path)

Detections: 13 Components: ['database', 'gateway', 'service']
Saved: /Users/vilella/Documents/fiap/pos_ia/iadt_fase05_final/assets/outputs/report.md
